# AI Presales Lab：免费 Colab GPU 的 QLoRA 实验

这个 notebook 将完成：

1. 检查 Colab GPU、CUDA 和显存；
2. 从 GitHub 拉取项目并安装不覆盖 Colab 自带 PyTorch 的训练依赖；
3. 重建并校验 case-level split 的微调数据；
4. 执行 QLoRA dry-run，再运行 Qwen2.5-0.5B-Instruct 的 TRL + PEFT 训练；默认训练 compact decision contract，避免让 0.5B 模型一次生成过大的嵌套方案；
5. 在同一份 held-out test split 上比较 base model 和 adapter；
6. 保存运行环境、配置、token audit、manifest、metrics、评估报告和 adapter。

Colab 的 GPU 类型、可用时间和资源配额会动态变化。这个实验只能证明本次 runtime 的结果，不能直接当成生产 SLA。不要上传客户资料、API Key、访问 Token 或未脱敏的 notebook 输出。

本项目的 canonical source 是 [GitHub/Lukesour/ai-presales-lab](https://github.com/Lukesour/ai-presales-lab)，每次请从该仓库的 `main` 分支重新打开本 notebook。不要从 Colab 的 Recent、Google Drive 副本或旧 tab 继续运行。

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone

# canonical source：每次从 GitHub main 打开本 notebook；不要使用 Drive/Recent 中的旧副本。
PROJECT_REPO_URL = 'https://github.com/Lukesour/ai-presales-lab.git'
PROJECT_BRANCH = 'main'

# True：训练/评估结束后把报告和 adapter 复制到 Google Drive；
# False：最后会生成 zip 并尝试下载，runtime 断开后 /content 文件会消失。
USE_DRIVE = True
DRIVE_RESULTS_DIR = Path('/content/drive/MyDrive/ai-presales-lab-results')
# 训练默认写本地磁盘，避免 Drive 挂载的文件系统拖慢或中断 Trainer 写 checkpoint。
# 若需要跨 runtime 中断恢复，可改为 True，但训练/保存可能更慢。
TRAIN_OUTPUT_ON_DRIVE = False
# 首次运行保持 False；如果显存 OOM，改成 True 使用 1 batch / 4096 tokens 的降级配置。
LOW_MEMORY = False
# 结构化答案较长；评估预算必须覆盖完整 JSON。JSON prefill 是可复现的输入控制，不是事后修复。
EVAL_MAX_NEW_TOKENS = 4096
JSON_PREFILL = True
# 0.5B 模型只学习短的决策对象；确定性的 Agent 保留完整 POC、模型策略和证据对象。
# 如需复现实验 1 的完整响应契约，可改为 'full'。
TARGET_PROFILE = 'compact'

# 模型公开可下载，不需要 Hugging Face Token。把缓存放在本地磁盘，避免 Drive I/O 拖慢训练。
os.environ['HF_HOME'] = '/content/huggingface-cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

if not PROJECT_REPO_URL.startswith('https://') or 'github.com/Lukesour/ai-presales-lab' not in PROJECT_REPO_URL:
    raise ValueError('PROJECT_REPO_URL 必须指向 canonical GitHub 仓库 https://github.com/Lukesour/ai-presales-lab.git。请从 GitHub main 重新打开 notebook。')

WORKDIR = Path('/content/ai-presales-qlora')
PROJECT_DIR = WORKDIR / 'ai-presales-lab'
REPORT_DIR = PROJECT_DIR / 'data' / 'results' / 'colab' / 'qlora'
WORKDIR.mkdir(parents=True, exist_ok=True)
print('workdir:', WORKDIR)
print('project:', PROJECT_DIR)

In [ ]:
# 可选但推荐：Colab runtime 是临时的，挂载 Drive 便于保存结果。
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    print('Drive output:', DRIVE_RESULTS_DIR)
else:
    print('Drive disabled; final zip must be downloaded before disconnecting the runtime.')

In [ ]:
# 拉取 canonical source。不要在 Colab 里把 Token 写进 URL，也不要混用旧项目目录。
def canonical_remote(url):
    return url.rstrip('/').removesuffix('.git')

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', PROJECT_BRANCH, PROJECT_REPO_URL, str(PROJECT_DIR)], check=True)
else:
    actual_remote = subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'remote', 'get-url', 'origin'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if canonical_remote(actual_remote) != canonical_remote(PROJECT_REPO_URL):
        raise RuntimeError(
            f'Colab 中已有错误项目目录：{actual_remote}；期望：{PROJECT_REPO_URL}。'
            '请执行 Runtime > Disconnect and delete runtime，然后从 GitHub main 重新打开 notebook。'
        )
    actual_branch = subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'branch', '--show-current'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if actual_branch != PROJECT_BRANCH:
        raise RuntimeError(
            f'Colab 中项目分支为 {actual_branch!r}，期望 {PROJECT_BRANCH!r}。'
            '请执行 Runtime > Disconnect and delete runtime，然后从 GitHub main 重新打开 notebook。'
        )
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', PROJECT_BRANCH], check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
print('repo:', subprocess.run(['git', 'remote', 'get-url', 'origin'], check=True, capture_output=True, text=True).stdout.strip())
print('branch:', subprocess.run(['git', 'branch', '--show-current'], check=True, capture_output=True, text=True).stdout.strip())
print('commit:', subprocess.run(['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip())

In [ ]:
# GPU preflight：nvidia-smi 仅用于展示，不作为训练的硬性依赖；真正以 PyTorch CUDA 为准。
nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    gpu_probe = subprocess.run([nvidia_smi], check=False, text=True, capture_output=True)
    print(gpu_probe.stdout)
    if gpu_probe.returncode != 0:
        print(gpu_probe.stderr)
else:
    print('提示：当前 PATH 中没有 nvidia-smi；它只用于展示，将继续检查 PyTorch CUDA。')

import torch
cuda_available = bool(torch.cuda.is_available())
cuda_diagnostics = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'cuda_available': cuda_available,
    'cuda_device_count': int(torch.cuda.device_count()),
    'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES'),
    'colab_gpu_env': os.environ.get('COLAB_GPU'),
    'nvidia_smi_available': bool(nvidia_smi),
}
if not cuda_available:
    print(json.dumps(cuda_diagnostics, ensure_ascii=False, indent=2))
    if torch.version.cuda is None:
        cause = '当前 PyTorch 看起来是 CPU-only 构建，或 CUDA runtime 未安装。'
    elif os.environ.get('CUDA_VISIBLE_DEVICES') == '':
        cause = 'CUDA_VISIBLE_DEVICES 为空，GPU 可能被当前 runtime 隐藏。'
    else:
        cause = '当前 runtime 没有附着可用 GPU，最常见原因是未选择 GPU、选择后未重新连接，或免费 GPU 暂时不可分配。'
    raise RuntimeError(
        'GPU preflight failed: torch.cuda.is_available()=False。'
        f'诊断：{cause} 请执行 Runtime > Change runtime type > GPU，保存后重新连接 runtime。'
        '不要在 Colab 中手动安装 CPU 版 torch；如果仍无法获得 GPU，请稍后重试或改用其他可用 runtime。'
    )
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
cuda_diagnostics.update({
    'gpu': gpu_name,
    'gpu_memory_gb': gpu_memory_gb,
    'bf16_supported': bool(torch.cuda.is_bf16_supported()),
    'compute_dtype': 'float16 for Tesla T4; the project config pins this explicitly',
})
print(json.dumps(cuda_diagnostics, ensure_ascii=False, indent=2))
if gpu_memory_gb < 12:
    print('提示：显存低于 12GB 时先保持默认小模型和 batch size；发生 OOM 再按 Runbook 降参。')

In [ ]:
# 安装项目和 Colab 专用训练依赖。finetune-colab 不主动覆盖 Colab 自带 torch。
REPORT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[finetune-colab]'], check=True)
# 本实验明确采用 bitsandbytes NF4，不使用 TorchAO。Colab 基础镜像可能预装旧版
# torchao；PEFT 会探测它并触发可选 dispatcher，因此将这个未使用的后端移除。
from importlib import metadata
try:
    torchao_version = metadata.version('torchao')
except metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None:
    print(f'removing unused optional torchao=={torchao_version} for bitsandbytes NF4 path')
    removal = subprocess.run(
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
        check=False, text=True, capture_output=True,
    )
    print((removal.stdout or '') + (removal.stderr or ''))
    if removal.returncode != 0:
        raise RuntimeError('无法移除 Colab 预装的 torchao；请删除 runtime 后重新运行 notebook。')
print('training dependencies installed; torchao intentionally absent for this QLoRA experiment')
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, stdout=(REPORT_DIR / 'pip-freeze.txt').open('w'))

In [ ]:
# 保存 runtime 元数据，后续简历只使用这次真实运行生成的数字。
REPORT_DIR.mkdir(parents=True, exist_ok=True)
runtime = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'repo': subprocess.run(['git', 'remote', 'get-url', 'origin'], check=True, capture_output=True, text=True).stdout.strip(),
    'branch': subprocess.run(['git', 'branch', '--show-current'], check=True, capture_output=True, text=True).stdout.strip(),
    'git_commit': subprocess.run(['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip(),
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu': gpu_name,
    'gpu_memory_gb': gpu_memory_gb,
    'bf16_supported': bool(torch.cuda.is_bf16_supported()),
}
(REPORT_DIR / 'runtime.json').write_text(json.dumps(runtime, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(runtime, ensure_ascii=False, indent=2))

In [ ]:
# 生成本次 runtime 使用的配置。训练默认写本地磁盘，结束后再复制到 Drive。
config_name = 'trl_qlora_colab_lowmem.json' if LOW_MEMORY else 'trl_qlora.json'
base_config_path = PROJECT_DIR / 'configs' / 'finetune' / config_name
runtime_config_path = WORKDIR / 'trl_qlora_runtime.json'
training_config = json.loads(base_config_path.read_text(encoding='utf-8'))
if USE_DRIVE and TRAIN_OUTPUT_ON_DRIVE:
    train_output_name = f'active-training-{TARGET_PROFILE}-lowmem' if LOW_MEMORY else f'active-training-{TARGET_PROFILE}'
    train_output_dir = DRIVE_RESULTS_DIR / train_output_name
else:
    train_output_dir = PROJECT_DIR / '.runtime' / 'models' / f'qwen2.5-0.5b-presales-{TARGET_PROFILE}-lora'
training_config['output_dir'] = str(train_output_dir)
runtime_config_path.write_text(json.dumps(training_config, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('training config:', runtime_config_path)
print('checkpoint/adapter output:', train_output_dir)

In [ ]:
# 重建并校验训练数据；生成过程是确定性的，不会读取客户数据。
env = os.environ.copy()
env['PYTHONPATH'] = 'src'
subprocess.run([
    sys.executable, 'scripts/build_finetune_dataset.py',
    '--target-profile', TARGET_PROFILE,
], check=True, env=env)
subprocess.run([sys.executable, 'scripts/check_finetune_dataset.py'], check=True, env=env)
# 用真实 tokenizer 审计完整 chat-template 长度；不要让 TRL 静默截断 JSON 尾部。
subprocess.run([
    sys.executable, 'scripts/audit_finetune_tokens.py',
    '--model', training_config['model_name_or_path'],
    '--directory', str(PROJECT_DIR / 'data' / 'finetuning'),
    '--max-length', str(training_config['training']['max_length']),
    '--strict',
    '--output', str(REPORT_DIR / 'token-audit.json'),
], check=True, env=env)
print((PROJECT_DIR / 'data' / 'finetuning' / 'manifest.json').read_text(encoding='utf-8'))

In [ ]:
# 先 dry-run，再训练。dry-run 失败时不要继续消耗 GPU。
subprocess.run([sys.executable, 'scripts/train_qlora.py', '--config', str(runtime_config_path), '--dry-run'], check=True, env=env)

In [ ]:
# 训练前校验当前 runtime config，避免误用旧的 Drive/混合精度配置。
active_config = json.loads(runtime_config_path.read_text(encoding='utf-8'))
active_training = active_config.get('training', {})
if active_training.get('trainer_precision') != 'fp32':
    raise RuntimeError('当前 runtime config 不是最新 T4 安全配置，请重新运行“生成本次 runtime 使用的配置”单元格。')
if int(active_training.get('max_length', 0)) < 4096:
    raise RuntimeError('当前 max_length 过小，可能截断结构化答案；请重新运行配置单元格。')
if Path(active_config['output_dir']) != train_output_dir:
    raise RuntimeError(f'训练输出目录与当前配置不一致：{active_config["output_dir"]} != {train_output_dir}；请重新运行配置单元格。')

# 若 runtime 中断，重新运行到本单元格前，先把 RESUME_CHECKPOINT 改成存在的 checkpoint-* 目录。
RESUME_CHECKPOINT = None
if USE_DRIVE and TRAIN_OUTPUT_ON_DRIVE:
    print('可用 checkpoint:', sorted(str(path) for path in train_output_dir.glob('checkpoint-*')))
run_env = dict(env)
run_env['ACCELERATE_MIXED_PRECISION'] = 'no'
train_command = [
    sys.executable, 'scripts/train_qlora.py',
    '--config', str(runtime_config_path),
]
if RESUME_CHECKPOINT:
    train_command += ['--resume-from-checkpoint', str(RESUME_CHECKPOINT)]

def run_logged(command, label):
    log_path = REPORT_DIR / f'{label}.log'
    result = subprocess.run(command, check=False, env=run_env, text=True, capture_output=True)
    combined = (result.stdout or '') + '\n' + (result.stderr or '')
    log_path.write_text(combined, encoding='utf-8')
    print(combined[-24000:])
    if result.returncode != 0:
        if USE_DRIVE:
            failed_dir = DRIVE_RESULTS_DIR / 'failed-runs' / datetime.now().strftime('%Y%m%d-%H%M%S')
            failed_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(log_path, failed_dir / log_path.name)
            print('完整失败日志已保存到:', failed_dir / log_path.name)
        raise RuntimeError(f'{label} failed with return code {result.returncode}; see {log_path}')
    return result

# 先做一次真正的模型/量化/优化器/梯度 dtype 冒烟测试，再启动完整训练。
if not RESUME_CHECKPOINT:
    run_logged(train_command + ['--smoke-test'], 'qlora-smoke-test')
run_logged(train_command, 'qlora-training')
adapter_dir = train_output_dir
assert (adapter_dir / 'adapter_config.json').exists(), f'adapter not found: {adapter_dir}'
print('adapter:', adapter_dir)

In [ ]:
# 在完全相同的 held-out test split 上评估 base 与 adapter。
# 每个评估在独立 subprocess 中运行，并保存完整 stdout/stderr，避免只看到 CalledProcessError。
test_file = 'data/finetuning/test.jsonl'
base_report = REPORT_DIR / 'base_eval.json'
adapter_report = REPORT_DIR / 'adapter_eval.json'
adapter_path = Path(adapter_dir)
if not (adapter_path / 'adapter_config.json').is_file():
    raise FileNotFoundError(f'缺少 adapter 配置: {adapter_path / "adapter_config.json"}')
if not any((adapter_path / name).is_file() for name in ('adapter_model.safetensors', 'adapter_model.bin')):
    raise FileNotFoundError(f'缺少 adapter 权重: {adapter_path}')

eval_env = dict(env)
eval_env['PYTHONUNBUFFERED'] = '1'
eval_env['TOKENIZERS_PARALLELISM'] = 'false'

def run_model_eval(label, extra_args, report_path):
    command = [
        sys.executable, 'scripts/evaluate_finetuned_model.py',
        '--model', 'Qwen/Qwen2.5-0.5B-Instruct',
        '--split', test_file,
        '--max-new-tokens', str(EVAL_MAX_NEW_TOKENS),
        '--contract-profile', TARGET_PROFILE,
    ]
    if JSON_PREFILL:
        command.append('--json-prefill')
    command += [*extra_args, '--output', str(report_path)]
    log_path = REPORT_DIR / f'{label}.log'
    result = subprocess.run(command, check=False, env=eval_env, text=True, capture_output=True)
    combined = (result.stdout or '') + '\n' + (result.stderr or '')
    log_path.write_text(combined, encoding='utf-8')
    print(combined[-24000:])
    if result.returncode != 0:
        if USE_DRIVE:
            failed_dir = DRIVE_RESULTS_DIR / 'failed-runs' / datetime.now().strftime('%Y%m%d-%H%M%S')
            failed_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(log_path, failed_dir / log_path.name)
            print('完整评估失败日志已保存到:', failed_dir / log_path.name)
        raise RuntimeError(f'{label} failed; 完整日志见 {log_path}')
    if not report_path.exists():
        raise RuntimeError(f'{label} completed without report: {report_path}')
    return report_path

run_model_eval('base-eval', [], base_report)
run_model_eval('adapter-eval', ['--adapter', str(adapter_path)], adapter_report)

def metrics(path):
    return json.loads(path.read_text(encoding='utf-8'))['metrics']
print('base:', json.dumps(metrics(base_report), ensure_ascii=False, indent=2))
print('adapter:', json.dumps(metrics(adapter_report), ensure_ascii=False, indent=2))

In [ ]:
# 保存到 Drive；如果不使用 Drive，则生成 zip 并下载。
run_name = datetime.now().strftime('%Y%m%d-%H%M%S')
bundle_dir = WORKDIR / f'qlora-result-{run_name}'
bundle_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(REPORT_DIR, bundle_dir / 'reports', dirs_exist_ok=True)
shutil.copytree(adapter_dir, bundle_dir / 'adapter', dirs_exist_ok=True)
shutil.copy(runtime_config_path, bundle_dir / 'trl_qlora.json')
shutil.copy(PROJECT_DIR / 'data' / 'finetuning' / 'manifest.json', bundle_dir / 'manifest.json')
archive = shutil.make_archive(str(bundle_dir), 'zip', root_dir=bundle_dir)
if USE_DRIVE:
    target = DRIVE_RESULTS_DIR / run_name
    shutil.copytree(bundle_dir, target, dirs_exist_ok=True)
    print('saved to Drive:', target)
else:
    from google.colab import files
    files.download(archive)
    print('downloaded:', archive)

## 实验结束后的记录

你需要从 `reports/runtime.json`、`reports/base_eval.json`、`reports/adapter_eval.json` 和 adapter 目录中整理：GPU 型号、显存、训练时间、train/eval loss、JSON parse rate、schema pass rate、policy pass rate、保守无证据数量和 OOM/中断情况；同时记录 `target_profile`，full 与 compact 不得混报。

只有当 base 与 adapter 使用同一个 test split、相同生成参数，并且结果已保存为 JSON 后，才把效果差异写进简历。训练没有跑完、输出无法解析或只在训练集上变好，都不能写成模型效果提升。